# Scraping Estadísticas FBref — TFG Guillermo Jiménez

Descarga las estadísticas de jugadores de las 5 grandes ligas europeas
desde FBref (sports-reference.com) y las guarda en un Excel estructurado.

**Fuente:** FBref publica tablas HTML con estadísticas por temporada. Cada liga tiene su propia URL.

**Output:** `Base de datos inicial.xlsx` → 5 hojas de jugadores + 5 hojas de equipos

In [1]:
!pip install -q requests beautifulsoup4 lxml openpyxl pandas

## Imports y configuración

In [2]:
import requests
import time
import random
import pandas as pd
from bs4 import BeautifulSoup
import os
import warnings
warnings.filterwarnings('ignore')

## Rutas y parámetros

In [3]:
import os
DB = '/content'

OUTPUT_PATH = os.path.join(DB, 'Base de datos inicial.xlsx')

# FBref bloquea scrapers sin User-Agent. Simulamos un navegador real.
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://fbref.com/',
}

# Delay entre peticiones: FBref limita a ~20 req/min sin bloquear
MIN_DELAY = 4.0
MAX_DELAY = 7.0

## URLs de las ligas

FBref organiza los datos por competición. Se usa la tabla `stats_standard` que incluye goles, asistencias, xG, minutos, etc.

In [4]:
LEAGUE_URLS = {
    'PL Players': {
        'url': 'https://fbref.com/en/comps/9/stats/players/Premier-League-Stats',
        'league': 'Premier League',
    },
    'LaLiga Players': {
        'url': 'https://fbref.com/en/comps/12/stats/players/La-Liga-Stats',
        'league': 'LaLiga',
    },
    'Serie A Players': {
        'url': 'https://fbref.com/en/comps/11/stats/players/Serie-A-Stats',
        'league': 'Serie A',
    },
    'Bundesliga Players': {
        'url': 'https://fbref.com/en/comps/20/stats/players/Bundesliga-Stats',
        'league': 'Bundesliga',
    },
    'Ligue 1 Players': {
        'url': 'https://fbref.com/en/comps/13/stats/players/Ligue-1-Stats',
        'league': 'Ligue 1',
    },
}

TEAM_URLS = {
    'PL Teams':          'https://fbref.com/en/comps/9/stats/squads/Premier-League-Stats',
    'LaLiga Teams':      'https://fbref.com/en/comps/12/stats/squads/La-Liga-Stats',
    'Serie A Teams':     'https://fbref.com/en/comps/11/stats/squads/Serie-A-Stats',
    'Bundesliga Teams':  'https://fbref.com/en/comps/20/stats/squads/Bundesliga-Stats',
    'Ligue 1 Teams':     'https://fbref.com/en/comps/13/stats/squads/Ligue-1-Stats',
}

## Función de scraping

FBref encierra algunas tablas en comentarios HTML para evitar scraping. Las descomentamos manualmente con BeautifulSoup.

In [5]:
def scrape_fbref_table(url, table_id=None):
    """
    Descarga una página de FBref y extrae la tabla de estadísticas principal.
    Devuelve: pd.DataFrame con las estadísticas, o None si hay error.
    """
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
    except requests.RequestException as e:
        print(f'  Error de conexión: {e}')
        return None

    if resp.status_code == 429:
        print('  Rate limit (429). Esperando 60 segundos...')
        time.sleep(60)
        resp = requests.get(url, headers=HEADERS, timeout=30)

    if resp.status_code != 200:
        print(f'  HTTP {resp.status_code}')
        return None

    soup = BeautifulSoup(resp.text, 'lxml')

    from bs4 import Comment
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        comment_soup = BeautifulSoup(comment, 'lxml')
        tables = comment_soup.find_all('table')
        if tables:
            for table in tables:
                if table_id is None or table.get('id') == table_id:
                    soup.body.append(table)

    if table_id:
        table = soup.find('table', {'id': table_id})
    else:
        table = soup.find('table', id=lambda x: x and 'stats_standard' in x)
        if not table:
            table = soup.find('table', class_='stats_table')

    if not table:
        print('  No se encontró la tabla de estadísticas')
        return None

    df = pd.read_html(str(table), header=1)[0]

    if 'Rk' in df.columns:
        df = df[df['Rk'] != 'Rk']
        df = df[df['Rk'].notna()]

    df = df.drop(columns=[c for c in df.columns if 'Matches' in str(c)], errors='ignore')

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(a), str(b)]).strip('_')
                      if b and b != a else str(a)
                      for a, b in df.columns]

    for col in df.columns:
        if col not in ['Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', 'Comp']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

## Ejecución del scraping

Descarga jugadores y equipos de las 5 ligas y guarda el Excel.

In [8]:
print('=' * 60)
print('  FBref Scraping — 5 Grandes Ligas (2023-24)')
print('=' * 60)

sheets = {}

# ── Jugadores ──────────────────────────────────────────────
for sheet_name, config in LEAGUE_URLS.items():
    print(f'\n{sheet_name} ({config["league"]})...')
    df = scrape_fbref_table(config['url'])
    if df is None:
        print(f'  Falló el scraping de {sheet_name}')
        continue
    df['League'] = config['league']
    df = df[df['Player'].notna()].copy()
    df = df[~df['Player'].astype(str).str.strip().isin(['', 'Player'])]
    print(f'  {len(df)} jugadores descargados')
    sheets[sheet_name] = df

# ── Equipos ────────────────────────────────────────────────
for sheet_name, url in TEAM_URLS.items():
    print(f'\n{sheet_name}...')
    df = scrape_fbref_table(url)
    if df is None:
        print(f'  Falló el scraping de {sheet_name}')
        continue
    print(f'  {len(df)} equipos descargados')
    sheets[sheet_name] = df

# ── Guardar Excel ──────────────────────────────────────────
print(f'\nGuardando "{OUTPUT_PATH}"...')

if not sheets:
    print('\n⚠️  No se pudo descargar ninguna hoja (todos los scrapings fallaron).')
    print('   Posibles causas: FBref bloqueó las peticiones (HTTP 403) o sin conexión.')
    print('   Sugerencia: espera unos minutos y vuelve a ejecutar, o usa una VPN.')
else:
    sheet_order = list(LEAGUE_URLS.keys()) + list(TEAM_URLS.keys())
    with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
        for sheet_name in sheet_order:
            if sheet_name in sheets:
                sheets[sheet_name].to_excel(writer, sheet_name=sheet_name, index=False)
                print(f'  Hoja "{sheet_name}" guardada')
    print(f'\nArchivo guardado: {OUTPUT_PATH}')
    print(f'  {len(sheets)} hojas creadas')

  FBref Scraping — 5 Grandes Ligas (2023-24)

PL Players (Premier League)...
  HTTP 403
  Falló el scraping de PL Players

LaLiga Players (LaLiga)...
  HTTP 403
  Falló el scraping de LaLiga Players

Serie A Players (Serie A)...
  HTTP 403
  Falló el scraping de Serie A Players

Bundesliga Players (Bundesliga)...
  HTTP 403
  Falló el scraping de Bundesliga Players

Ligue 1 Players (Ligue 1)...
  HTTP 403
  Falló el scraping de Ligue 1 Players

PL Teams...
  HTTP 403
  Falló el scraping de PL Teams

LaLiga Teams...
  HTTP 403
  Falló el scraping de LaLiga Teams

Serie A Teams...
  HTTP 403
  Falló el scraping de Serie A Teams

Bundesliga Teams...
  HTTP 403
  Falló el scraping de Bundesliga Teams

Ligue 1 Teams...
  HTTP 403
  Falló el scraping de Ligue 1 Teams

Guardando "/content/Base de datos inicial.xlsx"...

⚠️  No se pudo descargar ninguna hoja (todos los scrapings fallaron).
   Posibles causas: FBref bloqueó las peticiones (HTTP 403) o sin conexión.
   Sugerencia: espera unos mi

El código no funciona en un principio ya que FBref bloquea peticiones de descarga de sus archivos, yo usé una VPN en local, pero seguramente no funcione el código como scraping hecho desde otro dispositivo.